# Titanic V5 — 回归本质：单模型 + 精简特征

## 从 V4 和 4 个参考 notebook 中学到的

### V4 的关键发现
- **6-model ensemble OOF (0.8541) < LGBM alone (0.8552)** → ensemble 没帮助
- **57 特征在 891 行上 → 15:1 样本/特征比** → 过拟合
- Ticket_Prefix 的 18 列 one-hot 中，多数类别 <30 样本 → 纯噪声
- CV-LB 差距 0.078（诚实 CV 下仍存在）→ 真实过拟合

### 4 个参考 notebook 的共同规律
| 特征 | 共同点 |
|------|--------|
| **模型** | 100% 使用单模型（不用 ensemble）|
| **特征数** | 全部 <20（Notebook 2 只用 8 个特征）|
| **调参** | 轻量或不用（CatBoost 固定参数达到 0.8227 CV）|
| **CV** | 全部使用 StratifiedKFold |
| **Ticket** | 全部丢弃 |

### V5 策略：向参考 notebook 对齐
> "在 Titanic 上，少就是多。891 行数据承载不了 57 个特征。"

| # | 改动 | 说明 |
|---|------|------|
| 1 | **LGBM 单模型** | 不用 ensemble |
| 2 | **StratifiedKFold** | 对齐所有参考 notebook |
| 3 | **33 特征**（从 57 砍到 33） | 去掉 Ticket_Prefix(18)、Cabin_Multiple(1)、Deck FG/T |
| 4 | **KNN Age imputation** | Shai Nisan (81.1%) 同款，k=10 |
| 5 | **100 次 Optuna + 保守参数** | depth≤7, min_child≥20 |
| 6 | **LOO Family_Surv_Rate** | 保留（V3→V4 +0.007 的有效特征）

In [ ]:
# [V5-NEW] V5: Single LGBM, reduced features, KNN Age, StratifiedKFold
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import optuna

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries imported.')

In [ ]:
# Load and merge
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
train['Source'] = 'train'
test['Source'] = 'test'
test_ids = test['PassengerId'].copy()

full = pd.concat([train, test], axis=0, ignore_index=True)
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Survival rate: {train["Survived"].mean():.2%}')

In [ ]:
# Feature 1: Title
full['Title'] = full['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
full['Title'] = full['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
full.loc[full['Title'].isin(rare), 'Title'] = 'Rare'
print(full['Title'].value_counts())

In [ ]:
# Features 2-4: FamilySize, IsAlone, Surname
full['FamilySize'] = full['SibSp'] + full['Parch'] + 1
full['IsAlone'] = (full['FamilySize'] == 1).astype(int)
full['Surname'] = full['Name'].str.split(',').str[0]

# [V5-NEW] FamilyGroup for LOO survival rate computation
# Using Surname + Fare (rounded) to identify families
full['FamilyGroup'] = full['Surname'] + '_' + full['Fare'].round(2).astype(str)
print(f'Unique families: {full["FamilyGroup"].nunique()}')
print(f'FamilySize distribution:')
print(full['FamilySize'].value_counts().sort_index())

In [ ]:
# Feature 5: Deck — [V5-NEW] consolidated to 3 groups (ABC, DE, Unknown)
deck_map = {'A': 'ABC', 'B': 'ABC', 'C': 'ABC',
            'D': 'DE',  'E': 'DE',
            'F': 'U',   'G': 'U',   'T': 'U'}  # FG and T → Unknown (too few samples)
full['Deck'] = full['Cabin'].str[0].map(deck_map).fillna('U')
full['Has_Cabin'] = full['Cabin'].notna().astype(int)
print(full['Deck'].value_counts())

In [ ]:
# [V5-NEW] KNN Age imputation (Shai Nisan's 81.1% technique)
# Use Pclass, Sex, Fare, SibSp, Parch to impute Age with k=10 neighbors

# Prepare features for KNN
age_features = ['Pclass', 'Fare', 'SibSp', 'Parch']
# Create a copy for imputation
impute_df = full[age_features].copy()
impute_df['Sex_code'] = full['Sex'].map({'male': 0, 'female': 1})

# Standardize for KNN
scaler = StandardScaler()
impute_scaled = scaler.fit_transform(impute_df)

# Fit KNN imputer on all data (no leakage — Age doesn't use Survived)
knn_imputer = KNNImputer(n_neighbors=10)
imputed = knn_imputer.fit_transform(
    np.column_stack([impute_scaled, full[['Age']].values])
)
full['Age'] = imputed[:, -1]

# Age-based features
bins = [0, 5, 12, 18, 35, 60, 100]
labels = [0, 1, 2, 3, 4, 5]
full['AgeGroup'] = pd.cut(full['Age'], bins=bins, labels=labels).astype(int)
full['Age*Class'] = full['Age'] * full['Pclass']

print(f'Age missing after KNN: {full["Age"].isnull().sum()}')
print(f'Age stats: mean={full["Age"].mean():.1f}, std={full["Age"].std():.1f}')
print(f'Age*Class range: {full["Age*Class"].min():.0f} - {full["Age*Class"].max():.0f}')

In [ ]:
# Features 8-12: Fare imputation, derived features
full['Fare'] = full.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full['Fare_log'] = np.log1p(full['Fare'])
full['FarePerPerson'] = full['Fare'] / full['FamilySize']
full['Ticket_Frequency'] = full.groupby('Ticket')['Ticket'].transform('count')

# [V5-NEW] Fare*Pclass interaction — how rich/poor relative to class
# Low fare in 1st class vs high fare in 3rd class = very different meanings
full['Fare*Pclass'] = full['Fare'] * full['Pclass']

print(f'Fare missing: {full["Fare"].isnull().sum()}')
print(f'Ticket_Frequency: {full["Ticket_Frequency"].min()}-{full["Ticket_Frequency"].max()}')

In [ ]:
# [V5-NEW] LOO Family_Surv_Rate (same as V4, proven effective)
train_mask = full['Source'] == 'train'
global_mean = full.loc[train_mask, 'Survived'].mean()

# Family rate (LOO)
family_stats = full[train_mask].groupby('FamilyGroup')['Survived'].agg(['sum', 'count'])
family_stats.columns = ['fs', 'fc']
full['fs'] = full['FamilyGroup'].map(family_stats['fs']).fillna(0)
full['fc'] = full['FamilyGroup'].map(family_stats['fc']).fillna(0)

full['Family_Surv_Rate'] = global_mean
m = full['fc'] > 1
full.loc[train_mask & m, 'Family_Surv_Rate'] = (
    (full.loc[train_mask & m, 'fs'] - full.loc[train_mask & m, 'Survived']) / (full.loc[train_mask & m, 'fc'] - 1))
full.loc[~train_mask & m, 'Family_Surv_Rate'] = full.loc[~train_mask & m, 'fs'] / full.loc[~train_mask & m, 'fc']

# Ticket rate (LOO)
ticket_stats = full[train_mask].groupby('Ticket')['Survived'].agg(['sum', 'count'])
ticket_stats.columns = ['ts', 'tc']
full['ts'] = full['Ticket'].map(ticket_stats['ts']).fillna(0)
full['tc'] = full['Ticket'].map(ticket_stats['tc']).fillna(0)

full['Ticket_Surv_Rate'] = global_mean
m = full['tc'] > 1
full.loc[train_mask & m, 'Ticket_Surv_Rate'] = (
    (full.loc[train_mask & m, 'ts'] - full.loc[train_mask & m, 'Survived']) / (full.loc[train_mask & m, 'tc'] - 1))
full.loc[~train_mask & m, 'Ticket_Surv_Rate'] = full.loc[~train_mask & m, 'ts'] / full.loc[~train_mask & m, 'tc']

full['Surv_Rate'] = full[['Family_Surv_Rate', 'Ticket_Surv_Rate']].max(axis=1)
full['Surv_Rate_Invalid'] = ((full['Family_Surv_Rate'] == global_mean) & (full['Ticket_Surv_Rate'] == global_mean)).astype(int)

full = full.drop(['fs', 'fc', 'ts', 'tc'], axis=1)

corr = full.loc[train_mask, 'Family_Surv_Rate'].corr(full.loc[train_mask, 'Survived'])
print(f'Corr(Family_Surv_Rate, Survived): {corr:.4f} (V4 was 0.26, V3 leaked was 0.89)')

In [ ]:
# Final preprocessing — [V5-NEW] fewer features, StratifiedKFold compatible
drop_cols = ['PassengerId', 'Name', 'Surname', 'FamilyGroup', 'Ticket', 'Cabin', 'Source']
full = full.drop(columns=[c for c in drop_cols if c in full.columns])

full['Embarked'] = full['Embarked'].fillna('S')
full['Sex'] = full['Sex'].map({'female': 1, 'male': 0})
full['Pclass'] = full['Pclass'].astype(str)
full['AgeGroup'] = full['AgeGroup'].astype(str)

# [V5-NEW] Only encode necessary categoricals (no Ticket_Prefix!)
cat_cols = ['Embarked', 'Pclass', 'Title', 'Deck', 'AgeGroup']
full = pd.get_dummies(full, columns=cat_cols, drop_first=False)

# Split
train_size = 891
feature_cols = [c for c in full.columns if c != 'Survived']
X = full.iloc[:train_size][feature_cols].copy()
y = full.iloc[:train_size]['Survived'].astype(int).copy()
X_test = full.iloc[train_size:][feature_cols].copy()

print(f'Features: {X.shape[1]} (V4 had 57, V5 target ~30-35)')
print(f'Train: {X.shape}, Test: {X_test.shape}')
print(f'NaN in X: {X.isnull().sum().sum()}, NaN in X_test: {X_test.isnull().sum().sum()}')

In [ ]:
# [V5-NEW] StratifiedKFold model comparison (aligned with all 4 reference notebooks)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'LightGBM':            lgb.LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, verbose=-1),
    'CatBoost':            CatBoostClassifier(iterations=200, learning_rate=0.1, depth=6,
                                              random_state=RANDOM_STATE, verbose=0),
}

results = {}
print(f'{"Model":25s} | {"CV Mean":>8s} | {"Std":>6s}')
print('-' * 48)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name:25s} | {scores.mean():8.4f} | {scores.std():6.4f}')

baseline = 1 - y.mean()
print(f'{"Baseline (all perish)":25s} | {baseline:8.4f}')
print()
print('[V5-NEW] Compare: V4 LGBM (GroupKFold): 0.8171 | V5 LGBM (StratifiedKFold): here')

In [ ]:
sorted_r = sorted(results.items(), key=lambda x: x[1]['mean'], reverse=True)
names = [r[0] for r in sorted_r]
means = [r[1]['mean'] for r in sorted_r]
stds = [r[1]['std'] for r in sorted_r]
colors = ['#2ecc71' if m == max(means) else '#3498db' for m in means]

plt.figure(figsize=(10, 4))
plt.barh(range(len(names)), means, xerr=stds, color=colors, alpha=0.8)
plt.yticks(range(len(names)), names)
plt.xlabel('CV Accuracy (StratifiedKFold 5-fold)')
plt.title('V5 Model Comparison — Reduced Features, StratifiedKFold')
plt.axvline(x=baseline, color='red', linestyle='--', label=f'Baseline ({baseline:.4f})')
plt.legend()
for i, (m, _) in enumerate(zip(means, stds)):
    plt.text(m + 0.002, i, f'{m:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# [V5-NEW] Optuna LGBM — 100 trials, conservative search space
# StratifiedKFold splits (aligned with reference notebooks)
fold_splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X, y))

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 7),  # [V5-NEW] Conservative: max 7
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 60),  # [V5-NEW] Higher minimum
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),  # [V5-NEW] Max 0.9 for regularization
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10),  # [V5-NEW] Higher minimum reg
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10),
        'random_state': RANDOM_STATE,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    scores = []
    for train_idx, val_idx in fold_splits:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        scores.append(accuracy_score(y_val, model.predict(X_val)))
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nBest LGBM CV:  {study.best_value:.4f}')
print(f'Best params: {study.best_params}')
print(f'V4 LGBM (GroupKFold): 0.8552')
print(f'V5 LGBM (StratifiedKFold, conservative): above')

In [ ]:
# [V5-NEW] Single LGBM OOF prediction (no ensemble!)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lgbm_model = lgb.LGBMClassifier(**study.best_params, random_state=RANDOM_STATE, verbose=-1)

# OOF predictions for calibration check
lgbm_oof = cross_val_predict(lgbm_model, X, y, cv=skf, method='predict_proba')[:, 1]
lgbm_acc = accuracy_score(y, (lgbm_oof >= 0.5).astype(int))

print(f'LGBM OOF accuracy: {lgbm_acc:.4f}')
print(f'V4 ensemble OOF: 0.8541 (6-model weighted avg)')
print(f'V4 LGBM OOF:     0.8552 (single model)')
print(f'V5 LGBM OOF:     {lgbm_acc:.4f} (single model, conservative tuning, fewer features)')

# OOF prediction distribution
print(f'\nOOF survival rate: {(lgbm_oof >= 0.5).mean():.2%}')
print(f'Training survival rate: {y.mean():.2%}')

In [ ]:
# [V5-NEW] Threshold tuning — find optimal threshold on OOF predictions
best_thresh = 0.5
best_acc = lgbm_acc

print('Threshold search on OOF:')
for t in np.arange(0.40, 0.60, 0.01):
    acc = accuracy_score(y, (lgbm_oof >= t).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thresh = t
    marker = ' <<<' if acc == best_acc else ''
    print(f'  threshold={t:.2f}: accuracy={acc:.4f}{marker}')

print(f'\nBest threshold: {best_thresh:.2f} (accuracy={best_acc:.4f})')
print(f'Default 0.5:     {lgbm_acc:.4f}')
if best_thresh != 0.5:
    print(f'Improvement:     +{best_acc - lgbm_acc:.4f}')

In [ ]:
# Train on full data and predict test set
lgbm_model.fit(X, y)
test_proba = lgbm_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thresh).astype(int)

print(f'Test survival rate: {test_pred.mean():.2%}')
print(f'Training survival rate: {y.mean():.2%}')
print(f'(Close match = well-calibrated)')

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': test_pred
})
submission.to_csv('../submissions/submission-v5.csv', index=False)
print('Saved: ../submissions/submission-v5.csv')
print(f'Shape: {submission.shape}')
print(f'Distribution: {submission["Survived"].value_counts().to_dict()}')
print()
print(submission.head(10).to_string(index=False))

In [ ]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': lgbm_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
top_n = min(20, len(importance))
plt.barh(range(top_n), importance['importance'].head(top_n), color='steelblue')
plt.yticks(range(top_n), importance['feature'].head(top_n))
plt.gca().invert_yaxis()
plt.xlabel('Feature Importance')
plt.title(f'LGBM Feature Importance (Top {top_n}) — V5 Single Model')
plt.tight_layout()
plt.show()

print(f'Total features: {len(importance)}')
print(f'Top 20:')
print(importance.head(20).to_string(index=False))
print(f'\nFeatures with importance < 5: {len(importance[importance["importance"] < 5])}')

In [ ]:
import os, pandas as pd

print('=' * 70)
print('VERSION COMPARISON')
print('=' * 70)

# Load all submissions
subs = {}
for v in ['v1', 'v2', 'v3', 'v4', 'v5']:
    p = f'../submissions/submission-{v}.csv'
    if os.path.exists(p):
        subs[v] = pd.read_csv(p)

gender = pd.read_csv('../data/gender_submission.csv')

print(f'{"Version":12s} {"Pred 0":>6s} {"Pred 1":>6s} {"Rate":>7s} {"Kaggle":>10s} {"Note"}')
print('-' * 70)
for v in ['v1', 'v2', 'v3', 'v4', 'v5']:
    if v not in subs:
        continue
    s = subs[v]
    vc = s['Survived'].value_counts()
    info = {
        'v1': ('0.75837', 'Default params, no one-hot'),
        'v2': ('0.75837', 'Bugs fixed'),
        'v3': ('0.77033', '+FamRate (CV leaked!)'),
        'v4': ('0.77751', 'LOO + honest CV + ensemble'),
        'v5': ('?', 'Single LGBM, KNN Age, fewer features'),
    }
    score, note = info[v]
    print(f'{v:12s} {vc.get(0,0):6d} {vc.get(1,0):6d} {s["Survived"].mean():6.2%}  {score:>10s}  {note}')

print(f'{"gender":12s} {gender["Survived"].value_counts().get(0,0):6d} {gender["Survived"].value_counts().get(1,0):6d} {gender["Survived"].mean():6.2%}  {"0.76555":>10s}  {"Baseline"}')

# Agreement matrix
print(f'\n--- Agreement between versions ---')
versions = [v for v in ['v1', 'v2', 'v3', 'v4', 'v5'] if v in subs]
for i, v1 in enumerate(versions):
    for v2 in versions[i+1:]:
        agree = (subs[v1]['Survived'] == subs[v2]['Survived']).mean()
        print(f'  {v1} vs {v2}: {agree:.1%}')

if 'v5' in subs:
    agree = (subs['v5']['Survived'] == gender['Survived']).mean()
    diff = (subs['v5']['Survived'] != gender['Survived']).sum()
    print(f'  V5 vs gender: {agree:.1%} ({diff} different)')

print(f'\n--- Score Prediction ---')
print(f'  V4:     0.77751 (reference point)')
print(f'  V4 CV:  0.8552 (LGBM OOF)')
print(f'  V4 LB:  0.77751 (CV-LB gap: 0.078)')
print(f'  V5 CV:  {lgbm_acc:.4f} (OOF, conservative tuning)')
print(f'')
print(f'  Expected V5 LB: 0.780 - 0.795')
print(f'  Rationale:')
print(f'    - Fewer features (33 vs 57) → less overfitting → better generalization')
print(f'    - KNN Age imputation → better age signal')
print(f'    - Conservative tuning (max_depth=7, min_child≥20) → less overfit')
print(f'    - Single model → no ensemble degradation')
print(f'    - Threshold tuning → optimizes for accuracy metric')
print(f'')
print(f'  Reference scores for context:')
print(f'    Notebook 4 CatBoost (single):   0.8227 CV')
print(f'    Shai Nisan (Family LOO+KNN):   0.8110 LB')
print(f'    Chris Deotte (Name only):       0.8182 LB')
print(f'    Top legitimate:                 ~0.8420 LB')